In [ ]:
import os
import cv2
import numpy as np
import torch
import timm
import torchvision.transforms as T
from tqdm import tqdm
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


class Dataset:
    def __init__(self, dataset_folder, transforms, fallback_dummy=None):
        self.files = os.listdir(dataset_folder)
        self.dataset_folder = dataset_folder
        self.transforms = transforms
        self.fallback_dummy = fallback_dummy

    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        file = self.files[idx]
        file_path = f"{self.dataset_folder}/{file}"
        try:
            # Try to open the image with PIL
            img = Image.open(file_path)
            img = img.convert('RGB')
        
        except (OSError, Image.UnidentifiedImageError) as pil_error:
            # Handle PIL error by trying to read the image with OpenCV
            try:
                print(f"PIL error for file {file}: {pil_error}. Trying OpenCV...")
                img_cv = cv2.imread(file_path)
                if img_cv is None:
                    raise ValueError("OpenCV could not read the file either.")
    
                # Convert the OpenCV image (BGR to RGB) and then to PIL
                img_rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
                img = Image.fromarray(img_rgb)
    
            except Exception as error:
                # Use fallback if OpenCV also fails
                print(f"Error for file {file}: {error}")
                if self.fallback_dummy is not None:
                    img_tensor = self.fallback_dummy.clone()
                    return img_tensor, file
                else:
                    raise pil_error
    
        # Apply transformations
        img_tensor = self.transforms(img)
        return img_tensor, file


# Define path to the dataset
dataset_dir = '/FungiTastic'
device = 'cuda'


# BEIT-384 Features & Logits
- Extract and save logits and features from BEIT-384 model.

In [ ]:
folders = {
    'full-500p-train': f'{dataset_dir}/dataset/FungiTastic/FungiTastic/train/500p',
    'full-500p-val': f'{dataset_dir}/dataset/FungiTastic/FungiTastic/val/500p',
    'full-500p-test': f'{dataset_dir}/dataset/FungiTastic/FungiTastic/test/500p',

    'mini-500p-train': f'{dataset_dir}/dataset/FungiTastic/FungiTastic-Mini/train/500p',
    'mini-500p-val': f'{dataset_dir}/dataset/FungiTastic/FungiTastic-Mini/val/500p',
    'mini-500p-test': f'{dataset_dir}/dataset/FungiTastic/FungiTastic-Mini/test/500p',
}

model_name = 'beit-384-full'
model_l = timm.create_model('hf-hub:BVRA/beit_base_patch16_384.in1k_ft_fungitastic_384', pretrained=True).to(device).eval()
model_f = timm.create_model('hf-hub:BVRA/beit_base_patch16_384.in1k_ft_fungitastic_384', pretrained=True).to(device).eval()
model_f.reset_classifier(0)
transforms = T.Compose([
    T.Resize((384, 384)), 
    T.ToTensor(), 
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for dataset_name, dataset_folder in folders.items():
    dataset = Dataset(dataset_folder, transforms, fallback_dummy=torch.randn(3, 384, 384))
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, num_workers=6, shuffle=False)

    data = {'features': [], 'logits': [], 'files': []}
    for img, files in tqdm(loader):
        with torch.no_grad():
            data['files'].append(np.array(files))

            features = model_f(img.to(device))
            data['features'].append(features.cpu())

            logits = model_l(img.to(device))
            data['logits'].append(logits.cpu())

    data['files'] = np.concatenate(data['files'])
    data['features'] = torch.cat(data['features'])
    data['logits'] = torch.cat(data['logits'])

    folder_save = f'features/{model_name}'
    os.makedirs(folder_save, exist_ok=True)
    torch.save(data, f"{folder_save}/{dataset_name}.pth")


# DINOv2 - Features
- Extract and save features from DINOv2 model.

In [ ]:
folders = {
    'full-500p-train': f'{dataset_dir}/dataset/FungiTastic/FungiTastic/train/500p',
    'full-500p-val': f'{dataset_dir}/dataset/FungiTastic/FungiTastic/val/500p',
    'full-500p-test': f'{dataset_dir}/dataset/FungiTastic/FungiTastic/test/500p',

    'mini-500p-train': f'{dataset_dir}/dataset/FungiTastic/FungiTastic-Mini/train/500p',
    'mini-500p-val': f'{dataset_dir}/dataset/FungiTastic/FungiTastic-Mini/val/500p',
    'mini-500p-test': f'{dataset_dir}/dataset/FungiTastic/FungiTastic-Mini/test/500p',
}


model_name = 'dinov2'
model_features = timm.create_model('vit_large_patch14_reg4_dinov2.lvd142m', pretrained=True).eval().to(device)

transforms = T.Compose([
    T.Resize((518, 518)), 
    T.ToTensor(), 
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

for dataset_name, dataset_folder in folders.items():
    dataset = Dataset(dataset_folder, transforms, fallback_dummy=torch.randn(3, 518, 518))
    loader = torch.utils.data.DataLoader(dataset, batch_size=64, num_workers=6, shuffle=False)

    data = {'features': [], 'files': []}
    for img, files in tqdm(loader):
        with torch.no_grad():
            features = model_features(img.to(device))
            data['features'].append(features.cpu())
            data['files'].append(np.array(files))

    data['features'] = torch.cat(data['features'])
    data['files'] = np.concatenate(data['files'])
    folder_save = f'features/{model_name}'
    os.makedirs(folder_save, exist_ok=True)
    torch.save(data, f"{folder_save}/{dataset_name}.pth")
